In this notebook, I use 10.5 million rows of [temperature readings from sensors](https://www.kaggle.com/mattpo/temperature-iot-on-gcp) to detect when which, if any, windows are open. Through exploratory analysis, I detected and [reported](https://www.kaggle.com/mattpo/temperature-iot-on-gcp/discussion/242457) what's likely an error in a manually labelled open window event (which led to false minority labels in a very imbalanced data set). Correcting this single entry improved performance in all three baseline models, and increased the F1 score on held out data from 25% to 46%. Feature engineering and ensembling the models further increased the score to 68%. Importantly, for the sensors that were placed near their respective windows, all open window events were detected.

<a id="overview"></a>
### Overview

* [Introduction](#introduction)
* [Explore data](#explore-data): iteratively explore and process data
  * [Summary statistics and plots](#summary-statistics-and-plots)
  * [Combined data](#combined-data): look at temperature and window state data together
  * [Manual data entry error](#manual-data-entry-error)
* [Pre-process data](#pre-process-data) (data sets: original, with error corrected, additionally with engineered features)
* [Fit and evaluate models](#fit-and-evaluate-models)
  * [Fit models](#fit-models): compare baseline models on three versions of the data; combine with a voting classifier
  * [Test models](#test-models)

<a id="introduction"></a>
### Introduction

Internet of Things (IoT) is a system of internet-connected devices, used in private and public sectors (e.g. in the context of [manufacturing](https://rapidminer.com/blog/iiot-implementations-in-manufacturing/), [agriculture](https://www.businessinsider.com/smart-farming-iot-agriculture) and [smart cities](https://www.businessinsider.com/iot-smart-city-technology)) as well as [in households](https://ec.europa.eu/competition-policy/system/files/2021-06/internet_of_things_preliminary_report.pdf) all over the world. With affordable sensors and components, hobbyist can start their own DIY projects, including an open window detection project described in these [three-part blog posts](https://blog.doit-intl.com/production-scale-iot-best-practices-implementation-with-gcp-part-1-3-44e2fa0e6554). The data set is made of two components:

1) Temperature readings from three IoT sensors (258\* and 270\* are close to their windows; 275\* is further away from a third window): `timestamp_utc` (date time), `timestamp_epoch` (integer), `temp_f` (float), `temp_c` (float), `device_id` (string)

2) Open events for the windows (outside of these manually entered events, all windows were closed): `DayPST` (date), `StartTimePST` (time), `EndTimePST` (time), `ObjectCode` (window number, integer), `ObjectName` (window name, string)

Real-world data can be messy, and as [Andrew Ng stated](https://youtu.be/06-AZXmwHjo?t=397), sometimes data-centric approaches result in bigger improvements than model-centric approaches. In this analysis, I focus on using data wrangling and visualisation to identify and resolve data issues as well as creating features with predictive power.

In [ ]:
# data exploration / preprocessing
import pandas as pd
from pandas.api.types import CategoricalDtype
import numpy as np
import seaborn as sns
from matplotlib import pyplot as plt, gridspec
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

# model fitting / evaluation
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier, VotingClassifier

from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import (accuracy_score, precision_score,
                             recall_score, f1_score, 
                             classification_report, plot_confusion_matrix)

In [ ]:
# 10.5M rows of raw temperature sensor data
temperature_data_orig = pd.read_csv("../input/temperature-iot-on-gcp/temperature_data/temperature.csv", parse_dates=["timestamp_utc"], converters = {"device_id": str})

# 29 periods when windows open
window_data_orig = pd.read_csv("../input/temperature-iot-on-gcp/temperature_data/window_opened_closed.csv")

In [ ]:
assert temperature_data_orig.isna().sum().sum() == 0, "Some temperature data missing."
assert window_data_orig.isna().sum().sum() == 0, "Some window data missing."

print(f"First few lines of temperature data: \n{temperature_data_orig.head(3)}\n")
print(f"First few lines of window data: \n{window_data_orig.head(3)}\n")

temp_no_dups = (temperature_data_orig.shape == temperature_data_orig.drop_duplicates().shape)
if not temp_no_dups:
    print("At least one duplicate row in temperature data.")

window_no_dups = (window_data_orig.shape == window_data_orig.drop_duplicates().shape)
if not window_no_dups:
    print("At least one duplicate row in window data.")

Back to [Overview](#overview)

<a id="explore-data"></a>
### Explore data

It looks like the two tables have datetime data in different time zones, so I should fix that and prepare both tables for joining. I'll also rename some columns and shorten device IDs for clarity.

The temperature data isn't in regular intervals (which is expected with IoT data) and contains duplicates. I'll remove the duplicates then resample the data using the median (downsampling a little in the process) to create two data frames (in long and wide forms):

In [ ]:
window_data = (
 window_data_orig
 .assign(
     # PST is roughly UTC -8
     start_dt = lambda x: pd.to_datetime(x.DayPST + " " + x.StartTimePST),
     end_dt = lambda x: pd.to_datetime(x.DayPST + " " + x.EndTimePST),
     start_dt_utc = lambda x: (x.start_dt.dt.tz_localize("US/Pacific")
                               .dt.tz_convert("UTC").dt.tz_convert(None)),
     end_dt_utc = lambda x: (x.end_dt.dt.tz_localize("US/Pacific")
                             .dt.tz_convert("UTC").dt.tz_convert(None)),
     mins_opened = lambda x: (x.end_dt_utc - x.start_dt_utc).dt.total_seconds() / 60
 )
 .rename(columns={"ObjectCode": "window_code", "ObjectName": "window_name"})
 .drop(["DayPST", "StartTimePST", "EndTimePST", "start_dt", "end_dt"], axis=1)
)

# for combining with sensor data later
window_data["ts_opened"] = [
    pd.date_range(
        row.start_dt_utc.round("10s"), row.end_dt_utc.round("10s"), freq="10s"
    ) for row in window_data.itertuples()
]

In [ ]:
window_data.head(3)

In [ ]:
resamp_temp_data = (temperature_data_orig
                    .assign(dev_id = lambda x: x["device_id"].str[0:3] + "*")
                    .drop(["timestamp_epoch", "temp_f"], axis=1)
                    .drop_duplicates()
                    .set_index("timestamp_utc")
                    .groupby("dev_id")
                    .resample("10s").mean()
                    .reset_index())

resamp_temp_wide_data = (resamp_temp_data
                         .pivot(index="timestamp_utc", columns="dev_id", 
                                values="temp_c"))

combined_data = (
    window_data
    .explode("ts_opened")
    [["ts_opened", "window_code"]]
    .merge(resamp_temp_wide_data.reset_index(), how="outer",
           left_on="ts_opened", right_on="timestamp_utc")
    .drop("ts_opened", axis=1)
    .fillna({"window_code": 0})
    # type changes when merge
    .assign(window_code = lambda x: x["window_code"].astype(int))
)

In [ ]:
print(f"""Long form of resampled data ({resamp_temp_data.shape[0]} rows):
{resamp_temp_data.head(3)}\n""")

print(f"""Wide form of resampled data ({resamp_temp_wide_data.shape[0]} rows):
{resamp_temp_wide_data.head(3)}\n""")

print("Combined data:")
combined_data.head(3)

In [ ]:
DEV_IDS = ["258*", "270*", "275*"]  # will be used often

Back to [Overview](#overview)

<a id="summary-statistics-and-plots"></a>
#### **Summary statistics and plots**

*window data:*

In [ ]:
figure, axes = plt.subplots(1,2, figsize=(10, 3))
sns.countplot(x="window_code", data=window_data, ax=axes[0])  # only 29 events
sns.boxplot(x="window_code", y="mins_opened", data=window_data, ax=axes[1]);  # outlier w1

There are only 29 window-opening events altogether, one of which lasted almost 14 hours! It's hard to imagine why, especially considering it was winter, and could potentially be a data issue. Let's see what happens when I exclude it:

In [ ]:
print(window_data.query("mins_opened < 800")  # exclude outlier
      .groupby("window_code")
      .agg({"mins_opened": ["mean", "median", "min", "max"]}))

sns.displot(
    window_data.query("mins_opened < 800"),  # exclude outlier
    x="mins_opened", col="window_code",
    hue=pd.cut(window_data["mins_opened"], [0, 2, 5, 15, 30, 60, 90, 120]),
    binwidth=15, multiple="stack", height=2, aspect=1.5
);

On average, windows were opened for 25-40 minutes (median) - still quite a while for winter. Even after excluding the outlier, 4 out of 29 events lasted over 1 hour! On the other end of the scale, 5 events lasted up to 5 minutes, two of which lasted up to 2 minutes.

*temperature data:*

In [ ]:
print(f"""Share of missing data: 
{resamp_temp_wide_data.isna().sum() / resamp_temp_wide_data.shape[0]}
""")

sns.heatmap(resamp_temp_wide_data.isnull(), cbar=False);

In [ ]:
for event, rough_start in zip(["First", "Second"], ["2020-12-24", "2021-01-13"]):
    start_na = (resamp_temp_wide_data[resamp_temp_wide_data.isna().sum(axis=1) == 3]
                .query(f"timestamp_utc > '{rough_start}'")
                .head(1).index[0].strftime("%Y-%m-%d %H:%M:%S"))
    
    print(f"{event} streak of NaNs in all sensors starts: {start_na}")

After resampling to 10s intervals, it's clear 12-16% of data for each device is missing. This is also confirmed in the blog post:
>rows of data where one or more sensors failed to record a value [were excluded]. There were many instances where the power went out — my entire house for an hour on Christmas Day, Roomba bumping into a sensor power cord, Maple knocking a sensor over... expect the unexpected!

The data here suggests the sensor data was missing for more than an hour on Christmas day, so perhaps the sensors had to be switched back on after the power outage.

In [ ]:
# prep data
resamp_temp_period_data = pd.cut(
    resamp_temp_data["timestamp_utc"], 
    bins=10,
    labels=["bin" + str(item) for item in range(1, 11)])
resamp_temp_wide_subset = resamp_temp_wide_data.sample(50_000)

# plots
fig = plt.figure(figsize=(12, 6))
gs = gridspec.GridSpec(2, 3, figure=fig)
ax1, ax2, ax3 = (plt.subplot(spec) for spec in [gs[0, 0], gs[0, 1:3], gs[1, :]])

sns.histplot(resamp_temp_wide_subset, bins=30, ax=ax1)

sns.boxplot(
    x=resamp_temp_period_data, y="temp_c", 
    hue="dev_id", hue_order=DEV_IDS,
    data=resamp_temp_data, ax=ax2
).set(xlabel="", ylabel="Temperature (°C)", ylim=(0, 25))

# time series with annotation for long streak of NaNs in all three sensors
resamp_temp_wide_subset.plot(ax=ax3)
plt.axvline(x=pd.to_datetime("2020-12-25 19:53:00"),
            color="red", linestyle="--", linewidth=0.7)
plt.axvline(x=pd.to_datetime("2021-01-14 08:24:00"),
            color="red", linestyle="--", linewidth=0.7);

The temperature detected by 275\* seems to be almost always higher than the other two, in line with the fact that it's far from the window. Notably, there are some sudden and brief drops in temperature for 258\* and 270\* that probably correspond to window-opening events, but this is not seen in 275\*.

All three time series generally share daily seasonality and there are clearly periods of missing data. Interestingly, 270\* seems to have higher temperature readings than 258\* at first, but the two start to converge around the start of January.

Back to [Overview](#overview)

<a id="combined-data"></a>
#### **Combined data**

Next, I look at the window and temperature data together and explored relationships between the labels and potential features.

First, it's clear this is a data set with very imbalanced classes, with window 1 open events (1.7%) being slightly more frequent than open events from the other two windows (roughly 0.5%):

In [ ]:
combined_data["window_code"].value_counts(normalize=True)  # really imbalanced data

In [ ]:
(combined_data
 .groupby("window_code")
 [DEV_IDS]
 .agg({dev_id: ["mean", "std", "min", "max"] for dev_id in DEV_IDS}))

In [ ]:
sns_col_pal = sns.color_palette().as_hex()  # to make colours match other plots

def hide_current_axis(*args, **kwds):
    plt.gca().set_visible(False)

g = (pd.concat([
    combined_data[["window_code"] + DEV_IDS].query("window_code == 0").sample(5000),
    combined_data[["window_code"] + DEV_IDS].query("window_code > 0")
])
 .pipe(sns.PairGrid, hue="window_code", height=1.7, aspect=1.1,
       palette={0: sns_col_pal[3], 2: sns_col_pal[0], 
                1: sns_col_pal[1], 3: sns_col_pal[2]})
 .map_diag(sns.histplot, bins=30, 
           element="step", fill=False, cumulative=True,
           # easier to compare different window_codes than count:
           stat="probability", common_norm=False)
 .map_upper(hide_current_axis)
 .map_lower(sns.scatterplot, alpha=0.1)
)

g.fig.legend(
    handles=g._legend_data.values(), 
    labels=g._legend_data.keys(), 
    loc="upper right", ncol=4
).set(title="window_code");

Summary statistics and plots grouped by device ID and window state suggest sensor 258\* and 270\* temperatures would be the best predictors for when windows 2 and 1 are open, respectively. For example, based on `window_code` for:
- temperatures below ~11°C in 258\* or below ~13°C in 270\* (scatter plot, cumulative histogram)
- lowest temperature readings and highest temperature variations (std) in 258\* and 270\* (summary statistics)
- lowest mean temperature in 258\* (summary statistics; curiously, I don't see the same pattern for 270\*/window 1)

Since raw temperature readings alone seems insufficient to make the correct predictions, especially at higher temperature ranges, I think temperature difference and speeds of temperature changes might be helpful.

In contrast to the other two sensors, 275\* statistics are relatively similar regardless of window state, and temperature patterns are similar for `window_code`s 3 and 0 (window 3 open and no windows open).

In [ ]:
## prep for plot
def add_dt_highlight(sns_plot, y_lims, start_dt, end_dt, ax_pos: int, color=None):
    color = sns_col_pal[ax_pos] if color is None else color
    # https://stackoverflow.com/a/52317485 for fill_between with FacetGrid
    sns_plot.axes[ax_pos].fill_betweenx(y=y_lims, x1=start_dt, x2=end_dt,
                                        color=color, alpha=0.2)
        
def add_open_periods(sns_plot, y_lims, window_df: pd.DataFrame, 
                     window_code: int, ax_pos: int, color=None):
    filtered_df = window_df.query("window_code == @window_code")
    
    for start, end in zip(filtered_df["start_dt_utc"], filtered_df["end_dt_utc"]):
        add_dt_highlight(g, y_lims, start, end, ax_pos, color)
        
resamp_temp_stats = (resamp_temp_data.groupby("dev_id")
                     .agg({"temp_c": ["mean", "std"], 
                           "timestamp_utc": ["min", "max"]}))

dev_wind_dict = {"258*": 2, "270*": 1, "275*": 3}


## plot
g = (sns.FacetGrid(resamp_temp_data, col="dev_id", hue="dev_id",
                   col_wrap=1, height=1.5, aspect=7, dropna=False)
     .map_dataframe(plt.plot,"timestamp_utc","temp_c")
     .set(ylabel="Temperature (°C)",
          xticks=pd.date_range(start="2020-12-26", periods=6, freq="W-SAT")))

# open window
temperature_y_lims = plt.gca().axes.get_ylim()
for window_num, ax_row in zip([2, 1, 3], [0, 1, 2]):
    add_open_periods(g, temperature_y_lims, window_data, window_num, ax_pos=ax_row)

# possible missing open window data..?
g.axes[1].fill_betweenx(
    y=temperature_y_lims, 
    x1=pd.to_datetime("2021-01-06 01:00:00"), x2=pd.to_datetime("2021-01-06 07:00:00"),
    color="gray", alpha=0.2
)

# add band of average temperature
temperature_x_lims = plt.gca().axes.get_xlim()
for row in range(0, 3):
    g.axes[row].fill_between(
        temperature_x_lims,
        resamp_temp_stats[("temp_c", "mean")][row] - \
            resamp_temp_stats[("temp_c", "std")][row],  
        resamp_temp_stats[("temp_c", "mean")][row] + \
            resamp_temp_stats[("temp_c", "std")][row], 
        color=sns_col_pal[row], alpha = 0.1)

# update titles
for dev_id, ax in g.axes_dict.items():
    ax.set_title(f"{dev_id} (likely closest to window {dev_wind_dict[dev_id]})")

Overlaying the window opening events (shown by vertical shaded regions) on the temperature time series confirms that the sudden temperature drops for devices 258\* and 270\* correspond to opening events of the suspected windows. For other window opening events, I don't see obvious correlations with change in temperature. I've also added a horizontal band covering the mean +/- standard deviation for each time series, hoping that it might be useful for feature engineering, but there seems to be temperatures outside of the band regardless of window states.

Curiously, two window 1 open periods overlap on 2021-01-04 (around 2:30 - 3:00 PM PST as well as around 8 AM - 9:45 PM PST). Again, it seems unreasonable that a window would be left open for 14 hours in winter, *and* that the temperature would be unaffected in all sensors. Additionally, there's one big drop in temperature for sensor 270\* the next day without a corresponding window opening event (shaded in grey). Since the window opening data was manually entered, these observations suggest there was a possibly a mistake in the data entry, and I should look closer at the window 1 opening events.

Back to [Overview](#overview)

<a id="manual-data-entry-error"></a>
#### **Manual data entry error**

Looking at each window 1 opening events:

In [ ]:
check_data = window_data.query("window_code == 1").copy()
check_data["expanded_ts_opened"] = [
    pd.date_range(
        row.start_dt_utc.round("10s") - pd.Timedelta(5, "minute"), 
        row.end_dt_utc.round("10s") + pd.Timedelta(5, "minute"), 
        freq="10s"
    ) for row in check_data.itertuples()
]
check_data = check_data.reset_index().assign(event_num = lambda x: x["index"] + 1)

g = (check_data
     .explode("expanded_ts_opened")
     [["expanded_ts_opened", "window_code", "event_num"]]
     .merge(resamp_temp_wide_data.reset_index(), how="outer",
            left_on="expanded_ts_opened", right_on="timestamp_utc")
     .pipe((sns.FacetGrid, "data"),
           col="event_num", col_wrap=4, sharex=False,height=3, aspect=1)
     .map_dataframe(plt.plot, "timestamp_utc", "270*"))

# add window opening and closing annotation
for ax, start, end in zip(g.axes.flat, check_data["start_dt_utc"], check_data["end_dt_utc"]):
    ax.axvline(x=start, color="red", linestyle="--", linewidth=0.7)
    ax.axvline(x=end, color="red", linestyle="--", linewidth=0.7)

# update titles
for event_num, ax in g.axes_dict.items():
    mins_open = round(check_data.query('event_num == @event_num')
                      ['mins_opened'].values[0])
    ax.set_title(f"Event {int(event_num)} ({mins_open} minutes)")
    
g.set_xticklabels(rotation=90).fig.tight_layout()

There were quick drops and rises in 270\* temperature readings corresponding to most window 1 opening and closing events. The exceptions were event 4 (where data was missing), and events 2, 3 and 6 (which are difficult to explain considering the temperature dropped at least 2°C in events 8 and 9, which were much shorter events, only 2 minutes). There's also a strange horizontal line in event 3 resulting from the duplicated data.

Importantly, if the 2021-01-04 7:57:23 AM - 9:46:00 PM PST 14-hour window 1 opening event was supposed to be a 2-hour 2021-01-**05** 7:57:23 **PM** - 9:46:00 PM PST (2021-01-06 3:57:23 - 5:46:00 AM UTC) event, it would resolve the overlapping window opening issue and fit with the temperature data very well:

In [ ]:
(combined_data
 .query("timestamp_utc > '2021-01-06 3:40:00' & timestamp_utc < '2021-01-06 6:20'")
 [["timestamp_utc", "270*"]]
 .plot("timestamp_utc", "270*"))

# potential window opening and closing events
plt.axvline(x=pd.to_datetime("2021-01-06 3:57:23"), color="red", linestyle="--", linewidth=0.7)
plt.axvline(x=pd.to_datetime("2021-01-06 5:46:00"), color="red", linestyle="--", linewidth=0.7);

Together, I found the evidence compelling enough to [contact the author](https://www.kaggle.com/mattpo/temperature-iot-on-gcp/discussion/242457) to ask for his opinion on my theory. In the mean time, for the rest of this analysis, I will assume this is a manual data entry error that should be corrected before the modelling steps.

Back to [Overview](#overview)

<a id="pre-process-data"></a>
### Pre-process data

In [ ]:
print("Total index length:", len(combined_data.index))
print("\nsplitting date time for roughly 80%/20% split:")
(combined_data
 .sort_values(by="timestamp_utc")
 .reset_index(drop=True)
 .iloc[round(len(combined_data.index) * 0.8), 1]
)

In [ ]:
SPLIT_DAY = pd.to_datetime("2021-01-30 00:00:00")  # round up

def split_data(combined_data_df, train_cols):
    train = (combined_data_df.query("timestamp_utc < @SPLIT_DAY")
             .sort_values("timestamp_utc").set_index("timestamp_utc"))
    test = (combined_data_df.query("timestamp_utc >= @SPLIT_DAY")
            .sort_values("timestamp_utc").set_index("timestamp_utc"))
    
    X_train, y_train = train[train_cols], train["window_code"]
    X_test, y_test = test[train_cols], test["window_code"]
    
    # number of non-0.0 labels important to CV..
    print(pd.concat(dict(
        train_count=y_train.value_counts(), 
        train_prop=y_train.value_counts(normalize=True).sort_index(),
        test_count=y_test.value_counts(),
        test_prop=y_test.value_counts(normalize=True).sort_index()
    ), axis = 1).sort_index())
    
    imp_freq = SimpleImputer(missing_values=np.nan, strategy="most_frequent")
    X_train = pd.DataFrame(imp_freq.fit_transform(X_train), columns = X_train.columns)
    X_test = pd.DataFrame(imp_freq.transform(X_test), columns = X_test.columns)
        
    return X_train, y_train, X_test, y_test

**Original data**

In [ ]:
# mismatch in class distrib, especially for w1 (10x) and w2 (5x)
X_train_orig, y_train_orig, X_test_orig, y_test_orig = split_data(combined_data, DEV_IDS)

assert ~pd.concat([X_train_orig, X_test_orig]).isnull().any().any(), \
    "At least one missing value."

**Edited data**: correct window open-state annotation

In [ ]:
update_info = pd.DataFrame({"start_dt_utc": pd.to_datetime("2021-01-06 3:57:23"),
              "end_dt_utc": pd.to_datetime("2021-01-06 5:46:00")},
             index=window_data[window_data.mins_opened > 800].index.astype(int))

edited_window_data = window_data.query("end_dt_utc < '2021-02-07 21:09:47'").copy()
edited_window_data.update(update_info)
edited_window_data = (edited_window_data
                      .assign(mins_opened = lambda x: ((x.end_dt_utc - x.start_dt_utc)
                                                       .dt.total_seconds() / 60)
                      ))
edited_window_data["ts_opened"] = [
    pd.date_range(
        row.start_dt_utc.round("10s"), row.end_dt_utc.round("10s"), freq="10s"
    ) for row in edited_window_data.itertuples()
]

edited_combined_data = (edited_window_data
                        .explode("ts_opened")
                        [["ts_opened", "window_code"]]
                        .merge(resamp_temp_wide_data.reset_index(), how="outer",
                               left_on="ts_opened", right_on="timestamp_utc")
                        .fillna({"window_code": 0})
                        .assign(window_code = lambda x: x["window_code"].astype(int))
                        .drop("ts_opened", axis=1))  # all windows closed unless annotated

print("Updated entry in edited data:")
edited_window_data.query("start_dt_utc == '2021-01-06 03:57:23'")

In [ ]:
g = (sns.FacetGrid(resamp_temp_data, col="dev_id", hue="dev_id",
                   col_wrap=1, height=1.5, aspect=7, dropna=False)
     .map_dataframe(plt.plot,"timestamp_utc","temp_c")
     .set(ylabel="Temperature (°C)"))

# open windows
temperature_y_lims = plt.gca().axes.get_ylim()
for window_num, ax_row in zip([2, 1, 3], [0, 1, 2]):
    add_open_periods(g, temperature_y_lims, edited_window_data, window_num, ax_pos=ax_row)

With the data entry error fixed, window 1 opening events are more aligned with 270\* temperatures (while other entries were unaffected).

In [ ]:
edited_combined_data.head(3)

In [ ]:
print("Mean temperatures:\n", edited_combined_data.groupby("window_code")[DEV_IDS].mean(), "\n")

overlap_dt = len(combined_data[["timestamp_utc", "window_code"]]
                 .groupby("timestamp_utc").count().query("window_code > 1").index)
print(f"Original data: {overlap_dt} time points with overlapping window opening annotations.")

overlap_dt2 = len(edited_combined_data[["timestamp_utc", "window_code"]]
                  .groupby("timestamp_utc").count().query("window_code > 1").index)
print(f"Edited data: {overlap_dt2} time points with overlapping window opening annotations.")

In the edited data set, the mean 270\* temperature was lowest when window 1 was opened, similar to the case for 258\* temperatures and window 2. Also, the multiple window 1 opening annotations was fixed, and share of `window_code` 1 events adjusted to 0.7% (in the training data) from 1.9% in the original data set:

In [ ]:
X_train_edited, y_train_edited, X_test_edited, y_test_edited = \
    split_data(edited_combined_data, DEV_IDS)

assert all(y_test_orig.reset_index(drop=True) == y_test_edited.reset_index(drop=True)), \
    "something went wrong with editing"
assert ~pd.concat([X_train_edited, X_test_edited]).isnull().any().any(), \
    "still at least one missing value"

**Enhanced data**: add engineered features

In [ ]:
# 30m rolling windows, result set to right edge
roll_av_temp_wide_data = (resamp_temp_wide_data
                          .rolling(180, min_periods=1)
                          .median()  
                          .reset_index()
                          .rename(columns={dev_id: dev_id[0:3] + "_ra" 
                                           for dev_id in DEV_IDS}))

roll_std_temp_wide_data = (resamp_temp_wide_data
                           .rolling(180, min_periods=1)
                           .std()
                           .reset_index()
                           .rename(columns={dev_id: dev_id[0:3] + "_rstd"
                                            for dev_id in DEV_IDS}))

roll_low_lim_wide_data = (roll_av_temp_wide_data.merge(roll_std_temp_wide_data)
 .assign(
     lower_258 = lambda x: x["258_ra"] - x["258_rstd"],
     lower_270 = lambda x: x["270_ra"] - x["270_rstd"],
     lower_275 = lambda x: x["275_ra"] - x["275_rstd"]
 ).drop(["258_ra", "270_ra", "275_ra", "258_rstd", "270_rstd", "275_rstd"], axis=1))

enhanced_combined_data = (
    edited_combined_data
    .merge(roll_low_lim_wide_data, how="left")
    .assign(
        time_since_start = lambda x: (x.timestamp_utc - pd.to_datetime("2020-12-22")),
        days_since_start = lambda x: x.time_since_start.dt.days,
        lower_lim_258 = lambda x: x["258*"] < x["lower_258"],
        lower_lim_270 = lambda x: x["270*"] < x["lower_270"],
        lower_lim_275 = lambda x: x["275*"] < x["lower_275"],
        diff_5m_258 = lambda x: x["258*"] - x["258*"].shift(periods=30),
        diff_5m_270 = lambda x: x["270*"] - x["270*"].shift(periods=30),
        diff_5m_275 = lambda x: x["275*"] - x["275*"].shift(periods=30)
    )
    .drop(["time_since_start"], axis=1)
)

In [ ]:
enhanced_combined_data.head(3)

In [ ]:
enhanced_train_cols = DEV_IDS + ["days_since_start"] + \
    [prefix + dev_num 
     for prefix in ["lower_lim_", "diff_5m_"]
     for dev_num in ["258", "270", "275"]]

X_train_enhanced, y_train_enhanced, X_test_enhanced, y_test_enhanced = \
    split_data(enhanced_combined_data, enhanced_train_cols)

assert all(y_test_orig.reset_index(drop=True) == y_test_enhanced.reset_index(drop=True)), \
    "something went wrong with editing"
assert ~pd.concat([X_train_enhanced, X_test_enhanced]).isnull().any().any(), \
    "still at least one missing value"

Finally, let's double check there's no collinear features in the data sets:

In [ ]:
fig = plt.figure(figsize=(6, 6))
gs = gridspec.GridSpec(3, 2, figure=fig)
ax1, ax2, ax3 = (plt.subplot(spec) for spec in [gs[0, 0], gs[0, 1:], gs[1:, :]])

plt.subplots_adjust(wspace=0.5, hspace=0.5)

(sns.heatmap(X_train_orig.corr(method="pearson"), ax=ax1, annot=True, fmt=".1f")
 .set(title="Original data set"))
(sns.heatmap(X_train_edited.corr(method="pearson"), ax=ax2, annot=True, fmt=".1f")
 .set(title="Edited data set"))
sns.heatmap(enhanced_combined_data[enhanced_train_cols].corr(), ax=ax3, annot=True,
            fmt=".1f").set(title="Enhanced data set");

Back to [Overview](#overview)

<a id="fit-and-evaluate-models"></a>
### Fit and evaluate models

**Evaluation metrics**

Because of the class imbalance, the default metric (accuracy) would be high even in a useless model like predicting the majority class (0) or a minority class (e.g. 1) for every data point. Instead, I'll use macro-averaged precision and recall for troubleshooting. Since F1 is the harmonic mean of precision and recall, it's a good summary metric which I'll use for comparing models:

In [ ]:
EVAL_METRICS = ["precision", "recall", "f1"]

In [ ]:
all_0_y_pred = np.repeat(0.0, y_test_orig.shape)
all_1_y_pred = np.repeat(1.0, y_test_orig.shape)

metrics_eg = []
for y_pred in [all_0_y_pred, all_1_y_pred]:
    alt_scores = {metric: 
                  globals()[metric + "_score"](y_test_orig, y_pred, 
                                               average="macro", zero_division=0)
                  for metric in EVAL_METRICS}
    metrics_eg.append({"accuracy": accuracy_score(y_test_orig, y_pred), **alt_scores})
    
pd.DataFrame(metrics_eg).reset_index().rename(columns={"index": "predicted y"})

The use of **macro** rather than micro or weighted averaging is also important. This method simply takes the mean of the metric for each class, making the scores for each minority class just as important as the scores for the majority class. For example, for the model always predicting the majority class:

In [ ]:
# macro averaging (unweighted mean - all classes equal regardless of the support of each class)
# micro averaging (sum the number of true positives / false negatives for each class)
# weighted averaging (weight the score by the support of the class)

print("averaging method")
for averaging_method in ["weighted", "micro", "macro"]:
    print(f"{averaging_method} F1 score:",
          round(f1_score(y_test_orig, all_0_y_pred, average=averaging_method), 2))

**Cross validation**

[StratifiedKFold](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.StratifiedKFold.html) is normally suitable for cross validation of imbalanced data sets because it guarantees the same proportion of classes in all folds. But it's problematic for our time series data, since the data points are not independent from each other. scikit-learn has [TimeSeriesSplit](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.TimeSeriesSplit.html), a cross-validator specifically for time series data that uses k folds as train set, and k+1th fold as test in a k-fold validation:

<img src="https://scikit-learn.org/stable/_images/sphx_glr_plot_cv_indices_007.png" width="350"/><img src="https://scikit-learn.org/stable/_images/sphx_glr_plot_cv_indices_010.png" width="350"/>

The downside (especially for this use-case) is for most validation folds, I won't have much training data to use. Since there are only 29 window-opening events, some with missing temperature readings, spread across three windows, it's important to check that the each fold has enough data from all four classes in both training and testing sets:

In [ ]:
def check_class_prop(combined_data_version: str, cv_splitter):
    X_train_ps, y_train_ps, X_test, y_test = \
        [globals()[f"{prefix}_{combined_data_version}"]
         for prefix in ["X_train", "y_train", "X_test", "y_test"]]
    
    prop_df_list = []
    for train_ind, val_ind in cv_splitter.split(X_train_ps, y_train_ps):
        y_train, y_val = y_train_ps.iloc[train_ind], y_train_ps.iloc[val_ind]
        fold_df = pd.concat(dict(
            train_count=y_train.value_counts(), 
            train_prop=y_train.value_counts(normalize=True).sort_index(),
            val_count=y_val.value_counts(),
            val_prop=y_val.value_counts(normalize=True).sort_index()
        ), axis = 1).sort_index()
        
        if fold_df.isnull().sum().sum():
            return "missing data point from at least one class in at least one fold"
        prop_df_list.append(fold_df)
        
    final_df = (pd.concat([prop_df_list[0].assign(fold = 1), 
                           prop_df_list[1].assign(fold = 2)])
                .reset_index()
                .rename(columns={"index": "class_label"})
                .assign(class_label = lambda x: x.class_label.map(int))
                .style.format("{:.3f}", subset=["train_prop", "val_prop"])
               )
    return final_df

In [ ]:
# check edited data set since there are fewer 1.0 window_code data points
check_class_prop("edited", TimeSeriesSplit())  # n_splits=5 by default

Using the default five splits, there's minority class data missing in either training or testing set in at least one fold. But I can adjust both the number of splits and the test size in each split, to make sure that it works for both the original and edited data sets:

In [ ]:
tss = TimeSeriesSplit(n_splits=2, test_size=78_000)

check_class_prop("edited", tss)

Back to [Overview](#overview)

<a id="fit-models"></a>
#### Fit models

First, I'll run a few models on all three sets of processed data (original, edited to resolve the manual data entry error, and further enhanced with engineered features) for some baseline metrics:

In [ ]:
cat_type = CategoricalDtype(categories=["orig", "edited", "enhanced"], ordered=True)

def gen_baseline_eval(classifier_name: str, combined_data_version: str):
    # build and evaluate model
    X_train_unsplit, y_train_unsplit, X_test, y_test = \
        [globals()[f"{prefix}_{combined_data_version}"] for prefix in
         ["X_train", "y_train", "X_test", "y_test"]]
    pipe = Pipeline(steps=[("scaler", StandardScaler()),
                           ("classifier", globals()[classifier_name])])
    
    cv_metrics = dict(precision=[], recall=[], f1=[])
    for train_index, val_index in tss.split(X_train_unsplit, y_train_unsplit):
        X_train, X_val = X_train_unsplit.iloc[train_index], X_train_unsplit.iloc[val_index]
        y_train, y_val = y_train_unsplit.iloc[train_index], y_train_unsplit.iloc[val_index]
        
        pipe.fit(X_train, y_train)
        y_pred = pipe.predict(X_val)
        
        for metric_type in EVAL_METRICS:
            metric_fun = globals()[f"{metric_type}_score"]
            (cv_metrics
             .get(metric_type)
             .append(np.array(metric_fun(y_val, y_pred, average=None, zero_division=0))))
            
    # macro average for each class and metric combination (more detailed for troubleshooting)
    metrics_array = [[np.mean(fold) for fold in zip(*cv_metrics.get(metric))]
                     for metric in EVAL_METRICS]
    metrics_data = {metric_name: array 
                    for metric_name, array in zip(EVAL_METRICS, metrics_array)}
    metrics_df = pd.DataFrame(metrics_data)
    
    
    # also macro average for overall summary (easier to compare)
    summary_metrics = [np.mean(label_metrics) for label_metrics in metrics_array]
    summary_df = pd.DataFrame([summary_metrics], columns=EVAL_METRICS).rename(index={0: "avg"})
    
    
    # make final_dfs easy to combine with each other
    final_df = (pd.concat([metrics_df, summary_df])
                .reset_index()
                .rename(columns={"index": "label"})
                .assign(model=classifier_name, data=combined_data_version)
                .assign(data = lambda x: x["data"].astype(cat_type))
               )
    
    return final_df


def fmt_baseline_eval(pd_df, labs_to_highlight=["avg"]):
    # default highlight best avg (overall score) only
    cols_to_highlight = [(metric, column) 
                         for column in labs_to_highlight 
                         for metric in EVAL_METRICS]

    fmted_df = (pd_df
     .pivot(index=["model", "data"], columns="label", values=["precision", "recall", "f1"])
     .style.highlight_max(subset=cols_to_highlight).format("{:.1%}")
    )
    
    return fmted_df

In [ ]:
# saga is fast with large data sets and compatible with multiclass
log_reg = LogisticRegression(solver="saga", max_iter=2000, random_state=0)
knn = KNeighborsClassifier()
rf = RandomForestClassifier(random_state=0)

In [ ]:
baseline_mods_tscv = pd.concat([
    gen_baseline_eval(model, data_set)
    for model in ["log_reg", "knn", "rf"]
    for data_set in ["orig", "edited", "enhanced"]
])

fmt_baseline_eval(baseline_mods_tscv)

For all three models, using the edited data resulted in a higher average precision, recall and F1 scores than when using the original data set. In all but one case, using the enhanced dataset further improved this. It also resulted in non-zero precision, recall and F1 scores for label 3 data points in two out of three models, which was never seen when using the other two data sets.

Next, I combined the three models in a weighted soft voting classifier. The resulting model gave a higher F1 score, suggesting the weaker models still complement the best-performing model:

In [ ]:
voting_clf = VotingClassifier(
    estimators=[
        ("log_reg", log_reg),  # highest F1 scores (keep scores from going too low)
        ("knn", knn),  # non-0 metrics for window 3 and second highest F1 scores
        ("rf", rf)  # non-0 metrics for window 3
    ],
    weights=[3, 2, 1],
    voting="soft") # uses probability rather than majority vote

pd.concat([
    gen_baseline_eval("voting_clf", data_set) 
    for data_set in ["orig", "edited", "enhanced"]
]).pipe(fmt_baseline_eval)

Back to [Overview](#overview)

<a id="test-models"></a>
#### Test models

In [ ]:
def test_models(clf_name:str, combined_data_version: str, ax_pos):
    pipe = Pipeline(steps=[("scaler", StandardScaler()), ("clf", globals()[clf_name])])
    X_train, y_train, X_test, y_test = \
        [globals()[f"{prefix}_{combined_data_version}"] 
         for prefix in ["X_train", "y_train", "X_test", "y_test"]]
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    f1 = round(100 * f1_score(y_test, y_pred, average="macro"))
    
    clf_type = "Ensemble clf" if clf_name == "voting_clf" else "Best baseline clf"
    (plot_confusion_matrix(pipe, X_test, y_test, ax=axs[ax_pos])
     .ax_.set_title(f"{clf_type} + {combined_data_version}\ndata (F1 score {f1}%):"))
    
    return y_pred

fig, axs = plt.subplots(1, 3, figsize=(12, 3))
plt.subplots_adjust(wspace=0.4)

test_models("log_reg", "orig", 0)
test_models("log_reg", "edited", 1)
final_y_pred = test_models("voting_clf", "enhanced", 2)  # assign results for next step

Fitting the best baseline classifier on the original data resulted in a disappointing model that predicted 0 for every data point, and an F1 score of 25% on held-out data. Correcting the single window open entry error increased the F1 score to 46%. Finally, feature engineering and ensembling the models further increased the score to 68%.

Focusing on how to improve on the best performing model, it's clear that inability of the model to predict any positive window 3 labels pulled down the overall F1 scores. In fact, **the macro averaged F1 score for labels 0-2 is 90%**:

In [ ]:
print(classification_report(
    y_true=y_test_enhanced, y_pred=final_y_pred,
     zero_division=0, labels=[0, 1, 2, 3],
    target_names=["All closed", "Window 1 open", "Window 2 open", "Window 3 open"]
))

For a better understanding, I summarised the true and false positive, as well as the false negative predictions:

In [ ]:
def conditions_to_df(conditions, cond_col_name):
    df = (conditions
          .to_frame()
          .rename(columns={"window_code": cond_col_name})
          .assign(
              cumsum = lambda x: x[cond_col_name].cumsum(),
              cumsum_lag2 = lambda x: x["cumsum"].shift(2),
              cumsum_lead1 = lambda x: x["cumsum"].shift(-1),
              streak_start = lambda x: (x["cumsum"] == x["cumsum_lag2"] + 1) & x[cond_col_name],
              streak_end = lambda x: (x["cumsum"] == x["cumsum_lead1"]) & x[cond_col_name],
              true_label = y_test_enhanced
          ))
    return df

def gen_cond_summary(orig_df, event_type):
    summary_df = pd.DataFrame({
        "start_dt_utc": orig_df.query("streak_start").reset_index()["timestamp_utc"],
        "start_tl": orig_df.query("streak_start").reset_index()["true_label"],
        "end_dt_utc": orig_df.query("streak_end").reset_index()["timestamp_utc"],
        "end_tl": orig_df.query("streak_end").reset_index()["true_label"]
    }, index=range(orig_df.query("streak_start").shape[0]))
    
    assert (summary_df["start_tl"] == summary_df["end_tl"]).all(), \
        "Start and end true labels don't match."
    
    summary_df = (summary_df
                  .drop("start_tl", axis=1)
                  .rename(columns={"end_tl": "window_code"}))
    print(f"{event_type} events:\n{summary_df}\n")
    
    return summary_df


preds_match = final_y_pred == y_test_enhanced
window_opened = y_test_enhanced != 0
preds_0 = final_y_pred == 0

true_pos = conditions_to_df((preds_match & window_opened), "is_tp")
false_pos = conditions_to_df((~preds_match & ~preds_0), "is_fp")
false_negs = conditions_to_df((~preds_match & preds_0), "is_fn")

true_pos_summary = gen_cond_summary(true_pos, "True positive")
false_pos_summary = gen_cond_summary(false_pos, "False positive")
false_negs_summary = gen_cond_summary(false_negs, "False negative")

In [ ]:
print("Vertical bands show false negatives (grey) and true positives (other colours)")
g = (sns.FacetGrid(resamp_temp_data.query("timestamp_utc >= @SPLIT_DAY"), 
                   col="dev_id", hue="dev_id",
                   col_wrap=1, height=1.5, aspect=7, dropna=False)
     .map_dataframe(plt.plot,"timestamp_utc","temp_c")
     .set(ylabel="Temperature (°C)"))

# add true positive and false negative annotations
temperature_y_lims = plt.gca().axes.get_ylim()
for window_num, ax_row in zip([2, 1, 3], [0, 1, 2]):
    add_open_periods(g, temperature_y_lims, true_pos_summary, window_num, ax_pos=ax_row)
    add_open_periods(g, temperature_y_lims, false_negs_summary, 
                     window_num, ax_pos=ax_row, color="grey")
    
# update titles
for dev_id, ax in g.axes_dict.items():
    ax.set_title(f"{dev_id} (likely closest to window {dev_wind_dict[dev_id]})")

Overall, all 5 open window events in the held out data were detected for windows 1 and 2, with at least one streak of open state detected for at least 10 time points in a row. In contrast, out of the 6 false positive events for these windows, 5 occurred in only 1-2 time points in a row. Notably, 5 out of 6 false positive events occurred directly before or after a true positive open window event. Since the task was to detect open windows in order to remind the user to close them, I think the predictions can be used in combination with a rule-based logic to reliably detect open events for these two windows.

Unfortunately, none of the window 3 opening events were detected. Since the sensor was placed much further away from its window compared to other sensors, the signal to noise ratio might be too low. If it's feasible, moving the sensor could be a simple solution. If not, maybe use of other sensors detecting change in humidity, carbon dioxide and/or outside temperature could help.